In [1]:
# 🧠 Import Machine Learning libraries
import pandas as pd
import numpy as np

# Scikit-learn for ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, classification_report

# For saving the model
import joblib

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ Machine Learning libraries imported!")
print("🌲 Ready to train Random Forest!")
print("📊 Scikit-learn ready for ML magic!")

C:\Users\hp\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


✅ Machine Learning libraries imported!
🌲 Ready to train Random Forest!
📊 Scikit-learn ready for ML magic!


In [2]:
# 📂 Load dataset with sentiment analysis
dataset_path = '../DATASET/dataset_with_sentiment.csv'

print("⏳ Loading dataset with sentiment analysis...")
df = pd.read_csv(dataset_path)

print(f"\n✅ Dataset loaded successfully!")
print(f"📊 Total rows: {len(df):,}")
print(f"📋 Total columns: {len(df.columns)}")

# Preview key columns
print("\n" + "=" * 80)
print("🔍 PREVIEW (with sentiment data):")
print("=" * 80)
df[['product_name', 'Category', 'Final_Price', 'Rating', 'Sentiment_Label', 'Sentiment_Polarity']].head()

# Check sentiment distribution
print(f"\n🎭 Sentiment Distribution:")
print(df['Sentiment_Label'].value_counts())

⏳ Loading dataset with sentiment analysis...

✅ Dataset loaded successfully!
📊 Total rows: 188,301
📋 Total columns: 15

🔍 PREVIEW (with sentiment data):

🎭 Sentiment Distribution:
Sentiment_Label
POSITIVE    173477
NEGATIVE     14824
Name: count, dtype: int64


In [3]:
# 🔧 Prepare features for Machine Learning
print("=" * 80)
print("🔧 PREPARING FEATURES FOR ML TRAINING")
print("=" * 80)

# Create encoders for categorical columns
print("\n1️⃣ Encoding categorical variables...")
le_category = LabelEncoder()
le_vendor = LabelEncoder()
le_sentiment = LabelEncoder()
le_price_cat = LabelEncoder()
le_rating_cat = LabelEncoder()

# Encode Category (Electronics → 0, Grocery → 1, etc.)
df['Category_Encoded'] = le_category.fit_transform(df['Category'].astype(str))
print(f"   ✅ Category encoded: {len(le_category.classes_)} unique values")

# Encode Vendor (Amazon → 0, Flipkart → 1, etc.)
df['Vendor_Encoded'] = le_vendor.fit_transform(df['vendors'].astype(str))
print(f"   ✅ Vendor encoded: {len(le_vendor.classes_)} unique values")

# Encode Sentiment (POSITIVE → 1, NEGATIVE → 0)
df['Sentiment_Encoded'] = le_sentiment.fit_transform(df['Sentiment_Label'].astype(str))
print(f"   ✅ Sentiment encoded: {len(le_sentiment.classes_)} unique values")

# Encode Price Category
df['Price_Category_Encoded'] = le_price_cat.fit_transform(df['Price_Category'].astype(str))
print(f"   ✅ Price_Category encoded: {len(le_price_cat.classes_)} unique values")

# Encode Rating Category
df['Rating_Category_Encoded'] = le_rating_cat.fit_transform(df['Rating_Category'].astype(str))
print(f"   ✅ Rating_Category encoded: {len(le_rating_cat.classes_)} unique values")

# Show encoding mappings
print("\n2️⃣ Encoding Mappings:")
print("-" * 50)
print(f"📁 Categories: {dict(zip(le_category.classes_, range(len(le_category.classes_))))}")
print(f"🏪 Vendors: {dict(zip(le_vendor.classes_, range(len(le_vendor.classes_))))}")
print(f"🎭 Sentiments: {dict(zip(le_sentiment.classes_, range(len(le_sentiment.classes_))))}")

# Select features for ML
print("\n3️⃣ Selecting features for ML training...")
features = [
    'Category_Encoded',
    'Vendor_Encoded',
    'Original_Price',
    'Rating',
    'Review_Count',
    'Sentiment_Encoded',
    'Sentiment_Polarity',
    'Savings_Ratio',
    'Price_Category_Encoded',
    'Rating_Category_Encoded'
]

# Targets (what we want to predict)
target_discount = 'Discount_Percentage'
target_price = 'Final_Price'

X = df[features]
y_discount = df[target_discount]
y_price = df[target_price]

print(f"   ✅ Features (X): {len(features)} columns")
print(f"   ✅ Target 1: Discount_Percentage")
print(f"   ✅ Target 2: Final_Price")

print("\n" + "=" * 80)
print("📊 FEATURES READY FOR ML!")
print("=" * 80)
print(f"📋 Feature columns: {features}")
print(f"📏 X shape: {X.shape}")
print(f"📏 y_discount shape: {y_discount.shape}")
print(f"📏 y_price shape: {y_price.shape}")

🔧 PREPARING FEATURES FOR ML TRAINING

1️⃣ Encoding categorical variables...
   ✅ Category encoded: 118 unique values
   ✅ Vendor encoded: 13 unique values
   ✅ Sentiment encoded: 2 unique values
   ✅ Price_Category encoded: 4 unique values
   ✅ Rating_Category encoded: 4 unique values

2️⃣ Encoding Mappings:
--------------------------------------------------
📁 Categories: {'A': 0, 'ALOGY M': 1, 'AST': 2, 'AY': 3, 'B': 4, 'BLUK': 5, 'BLUKS BX': 6, 'BM': 7, 'BOLD STITCH BLUK': 8, 'BOYA  BY': 9, 'BOYA BY': 10, 'BOYA BY M': 11, 'BOYA M': 12, 'BOYA O': 13, 'BOYA W': 14, 'Beauty': 15, 'Beverages': 16, 'Books': 17, 'C': 18, 'CG': 19, 'CK': 20, 'COMPUTER LAPTOP L': 21, 'COMPUTER LAPTOP PC D': 22, 'Car&Motorbike': 23, 'Clothing': 24, 'D': 25, 'DM': 26, 'DXNICE DX': 27, 'DZ': 28, 'Dairy': 29, 'E': 30, 'EMEET': 31, 'ENRG': 32, 'ENRG K': 33, 'Electronics': 34, 'F': 35, 'FIFINE A': 36, 'Fashion': 37, 'Fitness Band': 38, 'Food': 39, 'Fruits & Vegetables': 40, 'G': 41, 'General': 42, 'Grocery': 43, '

In [4]:
# 📊 Split data into training and testing sets
print("=" * 80)
print("📊 SPLITTING DATA INTO TRAIN/TEST SETS")
print("=" * 80)

# Split for Discount prediction
X_train, X_test, y_disc_train, y_disc_test = train_test_split(
    X, y_discount, 
    test_size=0.2,      # 20% for testing, 80% for training
    random_state=42     # For reproducible results
)

# Split for Price prediction (same X split)
_, _, y_price_train, y_price_test = train_test_split(
    X, y_price,
    test_size=0.2,
    random_state=42
)

print(f"\n📊 DATA SPLIT SUMMARY:")
print(f"   🎓 Training set: {len(X_train):,} rows (80%)")
print(f"   🧪 Testing set:  {len(X_test):,} rows (20%)")

print(f"\n📏 Shape Details:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")
print(f"   y_disc_train shape: {y_disc_train.shape}")
print(f"   y_disc_test shape: {y_disc_test.shape}")
print(f"   y_price_train shape: {y_price_train.shape}")
print(f"   y_price_test shape: {y_price_test.shape}")

print("\n" + "=" * 80)
print("✅ DATA SPLIT COMPLETE - READY TO TRAIN!")
print("=" * 80)

📊 SPLITTING DATA INTO TRAIN/TEST SETS

📊 DATA SPLIT SUMMARY:
   🎓 Training set: 150,640 rows (80%)
   🧪 Testing set:  37,661 rows (20%)

📏 Shape Details:
   X_train shape: (150640, 10)
   X_test shape: (37661, 10)
   y_disc_train shape: (150640,)
   y_disc_test shape: (37661,)
   y_price_train shape: (150640,)
   y_price_test shape: (37661,)

✅ DATA SPLIT COMPLETE - READY TO TRAIN!


In [5]:
# 🌲 Train Random Forest Models
print("=" * 80)
print("🌲 TRAINING RANDOM FOREST MODELS")
print("=" * 80)
print("\n⏳ This will take 3-7 minutes... Grab a coffee! ☕")

# ============================================================
# MODEL 1: DISCOUNT PREDICTOR
# ============================================================
print("\n" + "=" * 80)
print("🏷️  MODEL 1: TRAINING DISCOUNT PREDICTOR")
print("=" * 80)

discount_model = RandomForestRegressor(
    n_estimators=100,       # 100 trees
    max_depth=20,           # Tree depth
    min_samples_split=10,   # Min samples to split
    random_state=42,        # Reproducibility
    n_jobs=-1,              # Use all CPU cores
    verbose=1               # Show progress
)

print("🌲 Planting 100 decision trees for DISCOUNT prediction...")
discount_model.fit(X_train, y_disc_train)
print("✅ Discount model trained!")

# ============================================================
# MODEL 2: PRICE PREDICTOR
# ============================================================
print("\n" + "=" * 80)
print("💰 MODEL 2: TRAINING PRICE PREDICTOR")
print("=" * 80)

price_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("🌲 Planting 100 decision trees for PRICE prediction...")
price_model.fit(X_train, y_price_train)
print("✅ Price model trained!")

print("\n" + "=" * 80)
print("🎉 BOTH MODELS TRAINED SUCCESSFULLY!")
print("=" * 80)
print("🌲 Total trees planted: 200 (100 + 100)")
print("📊 Ready for predictions!")

🌲 TRAINING RANDOM FOREST MODELS

⏳ This will take 3-7 minutes... Grab a coffee! ☕

🏷️  MODEL 1: TRAINING DISCOUNT PREDICTOR
🌲 Planting 100 decision trees for DISCOUNT prediction...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    7.8s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   18.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


✅ Discount model trained!

💰 MODEL 2: TRAINING PRICE PREDICTOR
🌲 Planting 100 decision trees for PRICE prediction...


[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   11.9s


✅ Price model trained!

🎉 BOTH MODELS TRAINED SUCCESSFULLY!
🌲 Total trees planted: 200 (100 + 100)
📊 Ready for predictions!


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   27.8s finished


In [6]:
# 📊 Evaluate Model Performance
print("=" * 80)
print("📊 EVALUATING MODEL PERFORMANCE")
print("=" * 80)

# ============================================================
# EVALUATE DISCOUNT MODEL
# ============================================================
print("\n" + "=" * 80)
print("🏷️  DISCOUNT MODEL EVALUATION")
print("=" * 80)

disc_pred = discount_model.predict(X_test)

disc_mae = mean_absolute_error(y_disc_test, disc_pred)
disc_rmse = np.sqrt(mean_squared_error(y_disc_test, disc_pred))
disc_r2 = r2_score(y_disc_test, disc_pred)

print(f"\n📈 Discount Prediction Metrics:")
print(f"   MAE (Mean Absolute Error): {disc_mae:.2f}")
print(f"   RMSE (Root Mean Squared Error): {disc_rmse:.2f}")
print(f"   R² Score (Accuracy): {disc_r2:.4f} ({disc_r2*100:.2f}%)")

# ============================================================
# EVALUATE PRICE MODEL
# ============================================================
print("\n" + "=" * 80)
print("💰 PRICE MODEL EVALUATION")
print("=" * 80)

price_pred = price_model.predict(X_test)

price_mae = mean_absolute_error(y_price_test, price_pred)
price_rmse = np.sqrt(mean_squared_error(y_price_test, price_pred))
price_r2 = r2_score(y_price_test, price_pred)

print(f"\n📈 Price Prediction Metrics:")
print(f"   MAE (Mean Absolute Error): ₹{price_mae:.2f}")
print(f"   RMSE (Root Mean Squared Error): ₹{price_rmse:.2f}")
print(f"   R² Score (Accuracy): {price_r2:.4f} ({price_r2*100:.2f}%)")

# ============================================================
# OVERALL SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("🎯 MODEL PERFORMANCE SUMMARY")
print("=" * 80)

print(f"\n🏷️  DISCOUNT MODEL:")
if disc_r2 > 0.8:
    print(f"   ✅ EXCELLENT! {disc_r2*100:.2f}% accuracy")
elif disc_r2 > 0.6:
    print(f"   ✅ GOOD! {disc_r2*100:.2f}% accuracy")
elif disc_r2 > 0.4:
    print(f"   ⚠️  AVERAGE. {disc_r2*100:.2f}% accuracy")
else:
    print(f"   ❌ NEEDS IMPROVEMENT. {disc_r2*100:.2f}% accuracy")
print(f"   📊 Error margin: ±{disc_mae:.2f}%")

print(f"\n💰 PRICE MODEL:")
if price_r2 > 0.8:
    print(f"   ✅ EXCELLENT! {price_r2*100:.2f}% accuracy")
elif price_r2 > 0.6:
    print(f"   ✅ GOOD! {price_r2*100:.2f}% accuracy")
elif price_r2 > 0.4:
    print(f"   ⚠️  AVERAGE. {price_r2*100:.2f}% accuracy")
else:
    print(f"   ❌ NEEDS IMPROVEMENT. {price_r2*100:.2f}% accuracy")
print(f"   📊 Error margin: ±₹{price_mae:.2f}")

# Sample predictions
print("\n" + "=" * 80)
print("🎯 SAMPLE PREDICTIONS VS ACTUAL (First 5 test products):")
print("=" * 80)

comparison = pd.DataFrame({
    'Actual_Discount': y_disc_test.values[:5],
    'Predicted_Discount': disc_pred[:5].round(2),
    'Actual_Price': y_price_test.values[:5].round(2),
    'Predicted_Price': price_pred[:5].round(2)
})
print(comparison.to_string(index=False))

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


📊 EVALUATING MODEL PERFORMANCE

🏷️  DISCOUNT MODEL EVALUATION

📈 Discount Prediction Metrics:
   MAE (Mean Absolute Error): 0.00
   RMSE (Root Mean Squared Error): 0.00
   R² Score (Accuracy): 1.0000 (100.00%)

💰 PRICE MODEL EVALUATION


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.2s



📈 Price Prediction Metrics:
   MAE (Mean Absolute Error): ₹13.41
   RMSE (Root Mean Squared Error): ₹274.11
   R² Score (Accuracy): 0.9987 (99.87%)

🎯 MODEL PERFORMANCE SUMMARY

🏷️  DISCOUNT MODEL:
   ✅ EXCELLENT! 100.00% accuracy
   📊 Error margin: ±0.00%

💰 PRICE MODEL:
   ✅ EXCELLENT! 99.87% accuracy
   📊 Error margin: ±₹13.41

🎯 SAMPLE PREDICTIONS VS ACTUAL (First 5 test products):
 Actual_Discount  Predicted_Discount  Actual_Price  Predicted_Price
            70.0                70.0        291.58           291.61
            20.0                20.0        974.94           974.97
             0.0                 0.0        127.57           127.07
            60.0                60.0        162.56           162.46
            20.0                20.0        343.07           343.10


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.4s finished


In [7]:
# 🎯 Feature Importance Analysis
print("=" * 80)
print("🎯 FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)
print("Which factors matter MOST for pricing decisions?\n")

# ============================================================
# DISCOUNT MODEL FEATURE IMPORTANCE
# ============================================================
print("=" * 80)
print("🏷️  DISCOUNT MODEL - Top Features")
print("=" * 80)

disc_importance = pd.DataFrame({
    'Feature': features,
    'Importance': discount_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nRank | Feature                    | Importance | Bar")
print("-" * 80)
for idx, row in disc_importance.iterrows():
    rank = disc_importance.index.get_loc(idx) + 1
    importance_pct = row['Importance'] * 100
    bar = '█' * int(importance_pct / 2)
    print(f"  {rank}  | {row['Feature']:25} | {importance_pct:6.2f}%  | {bar}")

# ============================================================
# PRICE MODEL FEATURE IMPORTANCE
# ============================================================
print("\n" + "=" * 80)
print("💰 PRICE MODEL - Top Features")
print("=" * 80)

price_importance = pd.DataFrame({
    'Feature': features,
    'Importance': price_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nRank | Feature                    | Importance | Bar")
print("-" * 80)
for idx, row in price_importance.iterrows():
    rank = price_importance.index.get_loc(idx) + 1
    importance_pct = row['Importance'] * 100
    bar = '█' * int(importance_pct / 2)
    print(f"  {rank}  | {row['Feature']:25} | {importance_pct:6.2f}%  | {bar}")

# ============================================================
# KEY INSIGHTS
# ============================================================
print("\n" + "=" * 80)
print("💡 KEY INSIGHTS")
print("=" * 80)

top_disc_feature = disc_importance.iloc[0]['Feature']
top_disc_importance = disc_importance.iloc[0]['Importance'] * 100

top_price_feature = price_importance.iloc[0]['Feature']
top_price_importance = price_importance.iloc[0]['Importance'] * 100

print(f"\n🏷️  MOST IMPORTANT for DISCOUNT:")
print(f"   ➡️  {top_disc_feature} ({top_disc_importance:.2f}%)")

print(f"\n💰 MOST IMPORTANT for PRICE:")
print(f"   ➡️  {top_price_feature} ({top_price_importance:.2f}%)")

print(f"\n🎯 Your model uses these patterns to make predictions!")

🎯 FEATURE IMPORTANCE ANALYSIS
Which factors matter MOST for pricing decisions?

🏷️  DISCOUNT MODEL - Top Features

Rank | Feature                    | Importance | Bar
--------------------------------------------------------------------------------
  1  | Savings_Ratio             | 100.00%  | █████████████████████████████████████████████████
  2  | Original_Price            |   0.00%  | 
  3  | Review_Count              |   0.00%  | 
  4  | Rating                    |   0.00%  | 
  5  | Price_Category_Encoded    |   0.00%  | 
  6  | Rating_Category_Encoded   |   0.00%  | 
  7  | Sentiment_Polarity        |   0.00%  | 
  8  | Category_Encoded          |   0.00%  | 
  9  | Sentiment_Encoded         |   0.00%  | 
  10  | Vendor_Encoded            |   0.00%  | 

💰 PRICE MODEL - Top Features

Rank | Feature                    | Importance | Bar
--------------------------------------------------------------------------------
  1  | Original_Price            |  94.25%  | ████████████████████

In [8]:
# 🔧 FIX DATA LEAKAGE - Re-train models properly!
print("=" * 80)
print("🔧 FIXING DATA LEAKAGE - RE-TRAINING MODELS")
print("=" * 80)

# Remove leaky features and target-related columns
print("\n⚠️  Removing data leakage features...")
print("   ❌ Savings_Ratio (directly calculates discount)")
print("   ❌ Price_Difference (used to calculate discount)")
print("   ❌ Final_Price from Discount features")

# NEW features - WITHOUT data leakage
clean_features = [
    'Category_Encoded',
    'Vendor_Encoded',
    'Original_Price',
    'Rating',
    'Review_Count',
    'Sentiment_Encoded',
    'Sentiment_Polarity',
    'Price_Category_Encoded',
    'Rating_Category_Encoded'
]

print(f"\n✅ Clean features: {len(clean_features)} features")
print(f"📋 Features: {clean_features}")

# Create new X with clean features
X_clean = df[clean_features]

# Re-split data with clean features
X_train_clean, X_test_clean, y_disc_train, y_disc_test = train_test_split(
    X_clean, y_discount, test_size=0.2, random_state=42
)

_, _, y_price_train, y_price_test = train_test_split(
    X_clean, y_price, test_size=0.2, random_state=42
)

print(f"\n📊 New data split:")
print(f"   Training: {len(X_train_clean):,} rows")
print(f"   Testing: {len(X_test_clean):,} rows")

# Re-train DISCOUNT model
print("\n" + "=" * 80)
print("🌲 RE-TRAINING DISCOUNT MODEL (without data leakage)")
print("=" * 80)

discount_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("⏳ Training... (2-3 minutes)")
discount_model.fit(X_train_clean, y_disc_train)
print("✅ Discount model re-trained!")

# Re-train PRICE model
print("\n" + "=" * 80)
print("🌲 RE-TRAINING PRICE MODEL (without data leakage)")
print("=" * 80)

price_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("⏳ Training... (2-3 minutes)")
price_model.fit(X_train_clean, y_price_train)
print("✅ Price model re-trained!")

# Update variables
X_train = X_train_clean
X_test = X_test_clean
features = clean_features

print("\n" + "=" * 80)
print("🎉 MODELS RE-TRAINED WITH CLEAN FEATURES!")
print("=" * 80)
print("📊 Now let's see REAL accuracy...")

🔧 FIXING DATA LEAKAGE - RE-TRAINING MODELS

⚠️  Removing data leakage features...
   ❌ Savings_Ratio (directly calculates discount)
   ❌ Price_Difference (used to calculate discount)
   ❌ Final_Price from Discount features

✅ Clean features: 9 features
📋 Features: ['Category_Encoded', 'Vendor_Encoded', 'Original_Price', 'Rating', 'Review_Count', 'Sentiment_Encoded', 'Sentiment_Polarity', 'Price_Category_Encoded', 'Rating_Category_Encoded']

📊 New data split:
   Training: 150,640 rows
   Testing: 37,661 rows

🌲 RE-TRAINING DISCOUNT MODEL (without data leakage)
⏳ Training... (2-3 minutes)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    9.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   22.4s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


✅ Discount model re-trained!

🌲 RE-TRAINING PRICE MODEL (without data leakage)
⏳ Training... (2-3 minutes)


[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   10.6s


✅ Price model re-trained!

🎉 MODELS RE-TRAINED WITH CLEAN FEATURES!
📊 Now let's see REAL accuracy...


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   28.9s finished


In [9]:
# 📊 Re-evaluate with clean features
print("=" * 80)
print("📊 REAL MODEL PERFORMANCE (without data leakage)")
print("=" * 80)

# Discount Model
disc_pred = discount_model.predict(X_test)
disc_mae = mean_absolute_error(y_disc_test, disc_pred)
disc_r2 = r2_score(y_disc_test, disc_pred)

print(f"\n🏷️  DISCOUNT MODEL (REAL):")
print(f"   R² Score: {disc_r2:.4f} ({disc_r2*100:.2f}%)")
print(f"   MAE: {disc_mae:.2f}%")

# Price Model
price_pred = price_model.predict(X_test)
price_mae = mean_absolute_error(y_price_test, price_pred)
price_r2 = r2_score(y_price_test, price_pred)

print(f"\n💰 PRICE MODEL (REAL):")
print(f"   R² Score: {price_r2:.4f} ({price_r2*100:.2f}%)")
print(f"   MAE: ₹{price_mae:.2f}")

# New feature importance
print("\n" + "=" * 80)
print("🎯 REAL FEATURE IMPORTANCE")
print("=" * 80)

print("\n🏷️  DISCOUNT MODEL - Top Features:")
disc_importance = pd.DataFrame({
    'Feature': features,
    'Importance': discount_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in disc_importance.iterrows():
    rank = disc_importance.index.get_loc(idx) + 1
    importance_pct = row['Importance'] * 100
    bar = '█' * int(importance_pct / 2)
    print(f"  {rank}. {row['Feature']:25} {importance_pct:6.2f}%  {bar}")

print("\n💰 PRICE MODEL - Top Features:")
price_importance = pd.DataFrame({
    'Feature': features,
    'Importance': price_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in price_importance.iterrows():
    rank = price_importance.index.get_loc(idx) + 1
    importance_pct = row['Importance'] * 100
    bar = '█' * int(importance_pct / 2)
    print(f"  {rank}. {row['Feature']:25} {importance_pct:6.2f}%  {bar}")

print("\n" + "=" * 80)
print("💡 NOW this is REAL ML learning!")
print("=" * 80)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s


📊 REAL MODEL PERFORMANCE (without data leakage)


[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s



🏷️  DISCOUNT MODEL (REAL):
   R² Score: 0.9313 (93.13%)
   MAE: 1.90%

💰 PRICE MODEL (REAL):
   R² Score: 0.9963 (99.63%)
   MAE: ₹64.10

🎯 REAL FEATURE IMPORTANCE

🏷️  DISCOUNT MODEL - Top Features:
  1. Review_Count               43.57%  █████████████████████
  2. Original_Price             25.28%  ████████████
  3. Price_Category_Encoded     17.73%  ████████
  4. Rating                      7.00%  ███
  5. Category_Encoded            4.90%  ██
  6. Sentiment_Polarity          0.80%  
  7. Rating_Category_Encoded     0.40%  
  8. Vendor_Encoded              0.21%  
  9. Sentiment_Encoded           0.10%  

💰 PRICE MODEL - Top Features:
  1. Original_Price             94.88%  ███████████████████████████████████████████████
  2. Price_Category_Encoded      2.95%  █
  3. Review_Count                0.95%  
  4. Rating                      0.83%  
  5. Category_Encoded            0.28%  
  6. Rating_Category_Encoded     0.07%  
  7. Sentiment_Polarity          0.03%  
  8. Sentiment_Enc

[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.5s finished


In [10]:
# 💾 Save ML Models for Production Use
import os

print("=" * 80)
print("💾 SAVING ML MODELS")
print("=" * 80)

# Create models directory
models_dir = '../backend/models'
os.makedirs(models_dir, exist_ok=True)
print(f"\n📁 Models directory: {models_dir}")

# Save Random Forest Models
print("\n1️⃣ Saving Random Forest Models...")
joblib.dump(discount_model, f'{models_dir}/discount_model.pkl')
print(f"   ✅ discount_model.pkl saved")

joblib.dump(price_model, f'{models_dir}/price_model.pkl')
print(f"   ✅ price_model.pkl saved")

# Save Label Encoders
print("\n2️⃣ Saving Label Encoders...")
joblib.dump(le_category, f'{models_dir}/le_category.pkl')
print(f"   ✅ le_category.pkl saved")

joblib.dump(le_vendor, f'{models_dir}/le_vendor.pkl')
print(f"   ✅ le_vendor.pkl saved")

joblib.dump(le_sentiment, f'{models_dir}/le_sentiment.pkl')
print(f"   ✅ le_sentiment.pkl saved")

joblib.dump(le_price_cat, f'{models_dir}/le_price_cat.pkl')
print(f"   ✅ le_price_cat.pkl saved")

joblib.dump(le_rating_cat, f'{models_dir}/le_rating_cat.pkl')
print(f"   ✅ le_rating_cat.pkl saved")

# Save features list
print("\n3️⃣ Saving features configuration...")
joblib.dump(features, f'{models_dir}/features.pkl')
print(f"   ✅ features.pkl saved")

# Save model metadata
metadata = {
    'discount_accuracy': disc_r2,
    'price_accuracy': price_r2,
    'discount_mae': disc_mae,
    'price_mae': price_mae,
    'features': features,
    'total_products': len(df),
    'training_rows': len(X_train),
    'testing_rows': len(X_test)
}
joblib.dump(metadata, f'{models_dir}/metadata.pkl')
print(f"   ✅ metadata.pkl saved")

# Verify saved files
print("\n" + "=" * 80)
print("📋 VERIFICATION - Files in backend/models/:")
print("=" * 80)

saved_files = os.listdir(models_dir)
for file in sorted(saved_files):
    file_path = f'{models_dir}/{file}'
    file_size = os.path.getsize(file_path) / 1024  # KB
    print(f"   ✅ {file:30} ({file_size:.2f} KB)")

print("\n" + "=" * 80)
print("🎉 ALL MODELS SAVED SUCCESSFULLY!")
print("=" * 80)
print(f"📊 Total files saved: {len(saved_files)}")
print("🚀 Ready for Flask backend integration!")

💾 SAVING ML MODELS

📁 Models directory: ../backend/models

1️⃣ Saving Random Forest Models...
   ✅ discount_model.pkl saved
   ✅ price_model.pkl saved

2️⃣ Saving Label Encoders...
   ✅ le_category.pkl saved
   ✅ le_vendor.pkl saved
   ✅ le_sentiment.pkl saved
   ✅ le_price_cat.pkl saved
   ✅ le_rating_cat.pkl saved

3️⃣ Saving features configuration...
   ✅ features.pkl saved
   ✅ metadata.pkl saved

📋 VERIFICATION - Files in backend/models/:
   ✅ discount_model.pkl             (74097.92 KB)
   ✅ features.pkl                   (0.18 KB)
   ✅ le_category.pkl                (1.59 KB)
   ✅ le_price_cat.pkl               (0.50 KB)
   ✅ le_rating_cat.pkl              (0.50 KB)
   ✅ le_sentiment.pkl               (0.48 KB)
   ✅ le_vendor.pkl                  (0.59 KB)
   ✅ metadata.pkl                   (0.35 KB)
   ✅ price_model.pkl                (139082.84 KB)

🎉 ALL MODELS SAVED SUCCESSFULLY!
📊 Total files saved: 9
🚀 Ready for Flask backend integration!


In [11]:
# 🎯 Test Predictions on NEW Products
print("=" * 80)
print("🎯 TESTING ML PREDICTIONS ON NEW PRODUCTS")
print("=" * 80)

def predict_product(category, vendor, original_price, rating, review_count, 
                    sentiment_label, sentiment_polarity):
    """Predict optimal price and discount for a new product"""
    
    # Determine Price Category
    if original_price <= 500:
        price_cat = 'Budget'
    elif original_price <= 2000:
        price_cat = 'Mid-Range'
    elif original_price <= 10000:
        price_cat = 'Premium'
    else:
        price_cat = 'Luxury'
    
    # Determine Rating Category
    if rating <= 2:
        rating_cat = 'Poor'
    elif rating <= 3.5:
        rating_cat = 'Average'
    elif rating <= 4.5:
        rating_cat = 'Good'
    else:
        rating_cat = 'Excellent'
    
    # Encode values
    cat_encoded = le_category.transform([category])[0]
    vendor_encoded = le_vendor.transform([vendor])[0]
    sent_encoded = le_sentiment.transform([sentiment_label])[0]
    price_cat_encoded = le_price_cat.transform([price_cat])[0]
    rating_cat_encoded = le_rating_cat.transform([rating_cat])[0]
    
    # Prepare input
    input_data = pd.DataFrame([[
        cat_encoded, vendor_encoded, original_price, rating, review_count,
        sent_encoded, sentiment_polarity, price_cat_encoded, rating_cat_encoded
    ]], columns=features)
    
    # Predict
    predicted_discount = discount_model.predict(input_data)[0]
    predicted_price = price_model.predict(input_data)[0]
    
    # Calculate savings
    savings = original_price - predicted_price
    actual_discount_pct = (savings / original_price) * 100
    
    return {
        'predicted_discount': round(predicted_discount, 2),
        'predicted_price': round(predicted_price, 2),
        'savings': round(savings, 2),
        'actual_discount_pct': round(actual_discount_pct, 2),
        'price_category': price_cat,
        'rating_category': rating_cat
    }

# ============================================================
# TEST CASES
# ============================================================

print("\n" + "=" * 80)
print("🧪 TEST 1: iPhone on Amazon (Positive Reviews)")
print("=" * 80)
result = predict_product(
    category='Electronics',
    vendor='Amazon',
    original_price=95000,
    rating=4.5,
    review_count=5000,
    sentiment_label='POSITIVE',
    sentiment_polarity=0.6
)
print(f"\n📱 Product: iPhone 14 Pro")
print(f"💰 Original Price: ₹95,000")
print(f"⭐ Rating: 4.5")
print(f"📝 Reviews: 5,000")
print(f"🎭 Sentiment: POSITIVE (+0.6)")
print(f"\n🎯 ML PREDICTIONS:")
print(f"   🏷️  Recommended Discount: {result['predicted_discount']}%")
print(f"   💰 Recommended Price: ₹{result['predicted_price']:,}")
print(f"   💵 Customer Savings: ₹{result['savings']:,}")
print(f"   📊 Price Category: {result['price_category']}")
print(f"   ⭐ Rating Category: {result['rating_category']}")

print("\n" + "=" * 80)
print("🧪 TEST 2: Grocery on Zomato (Negative Reviews)")
print("=" * 80)
result = predict_product(
    category='Grocery',
    vendor='Zomato',
    original_price=500,
    rating=2.5,
    review_count=800,
    sentiment_label='NEGATIVE',
    sentiment_polarity=-0.4
)
print(f"\n🛒 Product: Generic Grocery Item")
print(f"💰 Original Price: ₹500")
print(f"⭐ Rating: 2.5")
print(f"📝 Reviews: 800")
print(f"🎭 Sentiment: NEGATIVE (-0.4)")
print(f"\n🎯 ML PREDICTIONS:")
print(f"   🏷️  Recommended Discount: {result['predicted_discount']}%")
print(f"   💰 Recommended Price: ₹{result['predicted_price']:,}")
print(f"   💵 Customer Savings: ₹{result['savings']:,}")

print("\n" + "=" * 80)
print("🧪 TEST 3: Snacks on Zepto (Positive Reviews)")
print("=" * 80)
result = predict_product(
    category='Snacks',
    vendor='Zepto',
    original_price=200,
    rating=4.0,
    review_count=1500,
    sentiment_label='POSITIVE',
    sentiment_polarity=0.5
)
print(f"\n🍪 Product: Premium Chips")
print(f"💰 Original Price: ₹200")
print(f"⭐ Rating: 4.0")
print(f"📝 Reviews: 1,500")
print(f"🎭 Sentiment: POSITIVE (+0.5)")
print(f"\n🎯 ML PREDICTIONS:")
print(f"   🏷️  Recommended Discount: {result['predicted_discount']}%")
print(f"   💰 Recommended Price: ₹{result['predicted_price']:,}")
print(f"   💵 Customer Savings: ₹{result['savings']:,}")

print("\n" + "=" * 80)
print("🎉 ML MODEL READY FOR PRODUCTION!")
print("=" * 80)
print("🚀 These predictions will be used in:")
print("   - Customer Dashboard (show recommended prices)")
print("   - Admin Dashboard (suggest optimal discounts)")
print("   - Flask Backend API")

🎯 TESTING ML PREDICTIONS ON NEW PRODUCTS

🧪 TEST 1: iPhone on Amazon (Positive Reviews)


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.



📱 Product: iPhone 14 Pro
💰 Original Price: ₹95,000
⭐ Rating: 4.5
📝 Reviews: 5,000
🎭 Sentiment: POSITIVE (+0.6)

🎯 ML PREDICTIONS:
   🏷️  Recommended Discount: 56.01%
   💰 Recommended Price: ₹66,020.04
   💵 Customer Savings: ₹28,979.96
   📊 Price Category: Luxury
   ⭐ Rating Category: Good

🧪 TEST 2: Grocery on Zomato (Negative Reviews)

🛒 Product: Generic Grocery Item
💰 Original Price: ₹500
⭐ Rating: 2.5
📝 Reviews: 800
🎭 Sentiment: NEGATIVE (-0.4)

🎯 ML PREDICTIONS:
   🏷️  Recommended Discount: 51.75%
   💰 Recommended Price: ₹280.38
   💵 Customer Savings: ₹219.62

🧪 TEST 3: Snacks on Zepto (Positive Reviews)

🍪 Product: Premium Chips
💰 Original Price: ₹200
⭐ Rating: 4.0
📝 Reviews: 1,500
🎭 Sentiment: POSITIVE (+0.5)

🎯 ML PREDICTIONS:
   🏷️  Recommended Discount: 53.42%
   💰 Recommended Price: ₹162.04
   💵 Customer Savings: ₹37.96

🎉 ML MODEL READY FOR PRODUCTION!
🚀 These predictions will be used in:
   - Customer Dashboard (show recommended prices)
   - Admin Dashboard (suggest optima

[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished


In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from datetime import datetime
from textblob import TextBlob

# Adjust paths relative to notebook location
MODELS_DIR = '../backend/models'
DATASET_PATH = '../DATASET/dataset_with_sentiment.csv'

# Load dataset and clean
df = pd.read_csv(DATASET_PATH)
df['Category'] = df['Category'].str.strip()

# Load models
price_model = joblib.load(f'{MODELS_DIR}/price_model.pkl')
discount_model = joblib.load(f'{MODELS_DIR}/discount_model.pkl')
le_category = joblib.load(f'{MODELS_DIR}/le_category.pkl')
le_vendor = joblib.load(f'{MODELS_DIR}/le_vendor.pkl')
le_sentiment = joblib.load(f'{MODELS_DIR}/le_sentiment.pkl')
le_price_cat = joblib.load(f'{MODELS_DIR}/le_price_cat.pkl')
le_rating_cat = joblib.load(f'{MODELS_DIR}/le_rating_cat.pkl')
features = joblib.load(f'{MODELS_DIR}/features.pkl')

# Filter categories if required
categories = ['Electronics', 'Grocery', 'Snacks', 'Beverages', 'Fruits & Vegetables', 'Fashion', 'Home & Living', 'Beauty', 'Books', 'Clothing']
df = df[df['Category'].isin(categories)]

# Sample data for speed
sample_df = df.sample(min(1000, len(df)), random_state=42)

actual_prices = []
predicted_prices = []
actual_sentiments = []
predicted_sentiments = []

# Prepare features and predict
for _, row in sample_df.iterrows():
    try:
        original_price = float(row['Original_Price'])
        rating = float(row['Rating'])
        review_count = int(row['Review_Count'])
        category = row['Category']
        vendor = row['vendors']
        
        sentiment_label = str(row['Sentiment_Label']).upper()
        sentiment_polarity = float(row['Sentiment_Polarity'])

        price_category = 'Budget' if original_price <= 500 else 'Mid-Range' if original_price <= 2000 else 'Premium' if original_price <= 10000 else 'Luxury'
        rating_category = 'Poor' if rating <= 2 else 'Average' if rating <= 3.5 else 'Good' if rating <= 4.5 else 'Excellent'

        cat_enc = le_category.transform([category])[0]
        vendor_enc = le_vendor.transform([vendor])[0]
        sentiment_enc = le_sentiment.transform([sentiment_label])[0]
        price_cat_enc = le_price_cat.transform([price_category])[0]
        rating_cat_enc = le_rating_cat.transform([rating_category])[0]

        input_data = pd.DataFrame([[
            cat_enc, vendor_enc, original_price, rating, review_count,
            sentiment_enc, sentiment_polarity, price_cat_enc, rating_cat_enc
        ]], columns=features)

        pred_price = float(price_model.predict(input_data)[0])
        actual_prices.append(float(row['Final_Price']))
        predicted_prices.append(pred_price)

        predicted_sentiment = 'POSITIVE' if sentiment_polarity > 0.1 else 'NEGATIVE' if sentiment_polarity < -0.1 else 'NEUTRAL'
        actual_sentiments.append(sentiment_label)
        predicted_sentiments.append(predicted_sentiment)

    except Exception as e:
        continue  # Skip problematic rows

# Regression metrics
mse = mean_squared_error(actual_prices, predicted_prices)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual_prices, predicted_prices)
r2 = r2_score(actual_prices, predicted_prices)

print(f"Regression Metrics:")
print(f"  MSE: {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  R²: {r2:.4f}")

# Classification metrics
labels = sorted(list(set(actual_sentiments + predicted_sentiments)))
cm = confusion_matrix(actual_sentiments, predicted_sentiments, labels=labels)
accuracy = accuracy_score(actual_sentiments, predicted_sentiments)
precision = precision_score(actual_sentiments, predicted_sentiments, average='weighted', zero_division=0)
recall = recall_score(actual_sentiments, predicted_sentiments, average='weighted', zero_division=0)
f1 = f1_score(actual_sentiments, predicted_sentiments, average='weighted', zero_division=0)

print(f"\nClassification Metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1 Score: {f1:.4f}")
print(f"  Confusion Matrix (labels = {labels}):")
print(cm)

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_job

Regression Metrics:
  MSE: 695110.2192
  RMSE: 833.7327
  MAE: 253.6390
  R²: 0.9683

Classification Metrics:
  Accuracy: 1.0000
  Precision: 1.0000
  Recall: 1.0000
  F1 Score: 1.0000
  Confusion Matrix (labels = ['NEGATIVE', 'POSITIVE']):
[[ 61   0]
 [  0 939]]


[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.0s finished
